In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

# Métodos

In [2]:
def delete_hdf_table(file_path, key):
    with pd.HDFStore(file_path, mode='a') as store:
        if f'/{key}' in store.keys():
            store.remove(f'/{key}')
            print(f"Table '{key}' removed from HDF5.")
        else:
            print(f"Table '{key}' not found in HDF5.")

def read_hdf(file_path, key):
    return pd.read_hdf(file_path, mode = 'r', key=key, encoding='utf-8')

def show_hdf_tables(file_path):
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print(f"Current tables in {file_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

def write_hdf_chunks(df, path, key, chunksize=10_000_000):
    with pd.HDFStore(path, mode='a', complevel=9, complib='blosc:lz4') as store: # Open in read+write mode to append
        for i in range(0, len(df), chunksize):
            chunk = df.iloc[i:i+chunksize]
            append_mode = i != 0 # Append after the first chunk to avoid overwriting data from previous chunks
            store.append(
                key,
                chunk,
                format='table',
                data_columns=True,
                min_itemsize={'gauge_code': 20},
                encoding='utf-8',
                append=append_mode
            )
            del chunk  # free memory
            print(f"Written rows {i} to {min(i+chunksize-1, len(df))}")

def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    """
    Read HDF file in chunks and concatenate into a single DataFrame.
    Only works if the HDF key is stored in 'table' format.
    
    Parameters:
    file_path (str): Path to the HDF5 file
    key (str): Key/table name to read
    chunksize (int): Number of rows per chunk
    
    Returns:
    pd.DataFrame: Concatenated DataFrame
    """
    try:
        with pd.HDFStore(file_path, mode='r+') as store:  # Changed to 'r+' (read/write) and automatically closes the store
            if store.get_storer(key).is_table:
                dfs = []
                i = 1
                for chunk in store.select(key, chunksize=chunksize):
                    dfs.append(chunk)
                    print(f"Chunk {i} with {len(chunk)} rows read.")
                    i += 1                
                if dfs:  # Check if any chunks were read
                    return pd.concat(dfs, ignore_index=True)
                else:
                    print("No data found or chunksize too large.")
                    return pd.DataFrame()
            else:
                # If not table format, read all at once
                print("Not in table format, reading all data at once.")
                return store.select(key)  # Convert to DataFrame
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return pd.DataFrame()
    except KeyError:
        print(f"Key {key} not found in file {file_path}.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading HDF file: {e}")
        return pd.DataFrame()

# Data processing

In [ ]:
cleaned_path = r'./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'

In [4]:
df_data = read_hdf_chunks(cleaned_path, key='table_data_outlier_filtered', chunksize=1_000_000)
df_data

Chunk 1 with 1000000 rows read.
Chunk 2 with 1000000 rows read.
Chunk 3 with 1000000 rows read.
Chunk 4 with 1000000 rows read.
Chunk 5 with 1000000 rows read.
Chunk 6 with 1000000 rows read.
Chunk 7 with 1000000 rows read.
Chunk 8 with 1000000 rows read.
Chunk 9 with 1000000 rows read.
Chunk 10 with 1000000 rows read.
Chunk 11 with 1000000 rows read.
Chunk 12 with 1000000 rows read.
Chunk 13 with 1000000 rows read.
Chunk 14 with 1000000 rows read.
Chunk 15 with 1000000 rows read.
Chunk 16 with 1000000 rows read.
Chunk 17 with 1000000 rows read.
Chunk 18 with 1000000 rows read.
Chunk 19 with 1000000 rows read.
Chunk 20 with 1000000 rows read.
Chunk 21 with 1000000 rows read.
Chunk 22 with 1000000 rows read.
Chunk 23 with 1000000 rows read.
Chunk 24 with 1000000 rows read.
Chunk 25 with 1000000 rows read.
Chunk 26 with 1000000 rows read.
Chunk 27 with 1000000 rows read.
Chunk 28 with 1000000 rows read.
Chunk 29 with 1000000 rows read.
Chunk 30 with 1000000 rows read.
Chunk 31 with 10000

,gauge_code,datetime,rain_mm
0,110018901A,2021-02-02,9.48
1,110018901A,2021-02-03,4.73
2,110018901A,2021-02-04,73.28
3,110018901A,2021-02-07,0.20
4,110018901A,2021-02-08,0.00
...,...,...,...
123590252,88690050,2023-12-27,0.00
123590253,88690050,2023-12-28,0.00
123590254,88690050,2023-12-29,2.00
123590255,88690050,2023-12-30,0.00


In [ ]:
def calculateQ2(df):
    # Filter rows where rain_mm >= 1 mm (wet days)
    df_wet_days = df[df['rain_mm'] >= 1.0].copy() 
    
    # Extract year and day of the week from datetime
    df_wet_days['year'] = df_wet_days['datetime'].dt.year
    df_wet_days['day_of_week'] = df_wet_days['datetime'].dt.dayofweek  # Monday=0, Sunday=6
    
    # Group by gauge_code, year, and day_of_week, then count occurrences
    df_grouped = df_wet_days.groupby(['gauge_code', 'year', 'day_of_week']).size().reset_index(name='count')

    df_pivot = df_grouped.pivot(index=['gauge_code', 'year']
                                , columns='day_of_week'
                                , values='count').fillna(0).reset_index()    

    # Calculate the coefficient of variation (CV)
    df_pivot['std'] = df_pivot[[0, 1, 2, 3, 4, 5, 6]].std(axis=1)
    df_pivot['mean'] = df_pivot[[0, 1, 2, 3, 4, 5, 6]].mean(axis=1)
    
    df_pivot['cv'] =  df_pivot.apply(lambda x: x['std'] / x['mean'] if x['mean'] != 0 else 1.0, axis=1)
    
    # Calculate the Q2 for each group
    df_pivot['q2_week'] = 100 - 100 * df_pivot['cv']
    df_pivot['q2_week'] = df_pivot['q2_week'].clip(lower=0, upper=100)  # Set values below 0 to 0
    
    return df_pivot

df_q2_week = calculateQ2(df_data)
df_q2_week

day_of_week,gauge_code,year,0,1,2,3,4,5,6,std,mean,cv,q2_week
0,00047000,1961,7.0,7.0,7.0,9.0,7.0,4.0,4.0,1.812654,6.428571,0.281968,71.803161
1,00047000,1962,1.0,2.0,3.0,3.0,3.0,3.0,3.0,0.786796,2.571429,0.305976,69.402386
2,00047000,1963,7.0,9.0,11.0,12.0,8.0,13.0,18.0,3.716117,11.142857,0.333498,66.650234
3,00047000,1964,10.0,9.0,11.0,9.0,12.0,12.0,10.0,1.272418,10.428571,0.122013,87.798731
4,00047002,1977,3.0,0.0,1.0,0.0,1.0,2.0,1.0,1.069045,1.142857,0.935414,6.458565
...,...,...,...,...,...,...,...,...,...,...,...,...,...
324341,S712,2021,8.0,9.0,7.0,12.0,12.0,6.0,10.0,2.340126,9.142857,0.255951,74.404870
324342,S713,2021,1.0,1.0,2.0,1.0,2.0,1.0,2.0,0.534522,1.428571,0.374166,62.583426
324343,S714,2021,12.0,4.0,5.0,8.0,7.0,9.0,8.0,2.636737,7.571429,0.348248,65.175174
324344,S715,2021,11.0,7.0,11.0,12.0,14.0,13.0,9.0,2.380476,11.000000,0.216407,78.359308


In [6]:
df_data['year'] = df_data['datetime'].dt.year
df_gauge_code = df_data[['gauge_code', 'year']].drop_duplicates().sort_values(by=['gauge_code', 'year'])
df_gauge_code = df_gauge_code.reset_index(drop=True)
df_gauge_code

,gauge_code,year
0,00047000,1961
1,00047000,1962
2,00047000,1963
3,00047000,1964
4,00047002,1977
...,...,...
345863,S713,2021
345864,S714,2021
345865,S715,2021
345866,S716,2021


In [7]:
df_q2_week = df_q2_week[['gauge_code', 'year', 'q2_week']]
df_q2_week = pd.merge(df_gauge_code, df_q2_week, on=['gauge_code', 'year'], how='left')
df_q2_week = df_q2_week.sort_values(by=['gauge_code', 'year']).fillna(0)
df_q2_week = df_q2_week.reset_index(drop=True)
df_q2_week

,gauge_code,year,q2_week
0,00047000,1961,71.803161
1,00047000,1962,69.402386
2,00047000,1963,66.650234
3,00047000,1964,87.798731
4,00047002,1977,6.458565
...,...,...,...
345863,S713,2021,62.583426
345864,S714,2021,65.175174
345865,S715,2021,78.359308
345866,S716,2021,77.324050


In [11]:
# df_q2_week.to_hdf('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'
#                   , key = 'table_q2_week'
#                   , encoding = 'utf-8'
#                   , mode='r+'
#                   , format='table'
#                   , append=False)

write_hdf_chunks(df_q2_week, cleaned_path
                 , key = 'table_q2_week'
                 , chunksize=10_000_000)
show_hdf_tables(cleaned_path)                          
df_q2_week

Written rows 0 to 345868
Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_p_availability
  - table_preclassif
  - table_q1_gaps
  - table_q2_week


,gauge_code,year,q2_week
0,00047000,1961,71.803161
1,00047000,1962,69.402386
2,00047000,1963,66.650234
3,00047000,1964,87.798731
4,00047002,1977,6.458565
...,...,...,...
345863,S713,2021,62.583426
345864,S714,2021,65.175174
345865,S715,2021,78.359308
345866,S716,2021,77.324050


In [12]:
df_teste = read_hdf(cleaned_path, 'table_q2_week')
df_teste

,gauge_code,year,q2_week
0,00047000,1961,71.803161
1,00047000,1962,69.402386
2,00047000,1963,66.650234
3,00047000,1964,87.798731
4,00047002,1977,6.458565
...,...,...,...
345863,S713,2021,62.583426
345864,S714,2021,65.175174
345865,S715,2021,78.359308
345866,S716,2021,77.324050
